# Payment Agent를 AgentCore Runtime에 배포

## 개요

Tutorial 01에서는 payment-enabled agent를 Notebook에서 로컬로 실행했습니다. 이 튜토리얼에서는 동일한 agent를
**AgentCore Runtime**에 배포하여 SigV4 auth와 HTTPS를 통해 호출할 수 있도록 합니다.

배포된 agent는 전용 execution role로 실행됩니다. **AgentCorePaymentsPlugin**은
402 response를 자동으로 처리하며 LLM은 payment API를 직접 호출하지 않습니다.

```
App Backend                          AgentCore Runtime
  │                                   ┌──────────────────────────┐
  │ create_session(budget=$0.50)      │  Payment Agent            │
  │                                   │  (execution role)         │
  │── invoke(session, instrument) ──►│  Plugin: ProcessPayment   │
  │                                   │  불가: CreateSession      │
  │◄── weather data + cost ─────────│  불가: budget override     │
  │                                   └──────────────────────────┘
  │ get_session(지출 확인)
```

### Agent 코드 작동 방식

`payment_agent.py`는 세 가지 pattern을 사용합니다.

1. **`BedrockAgentCoreApp` + `@app.entrypoint`** — 표준 AgentCore Runtime service contract
2. **Payload 기반 config** — 모든 payment context(manager ARN, session, instrument)를 invocation payload에서 가져옵니다. 따라서 agent가 stateless 상태로 유지됩니다.
3. **`AgentCorePaymentsPlugin`** — HTTP 402 response를 intercept하고 session budget 내에서 `ProcessPayment`를 자동으로 호출

### 튜토리얼 Flow

```
dependency 설치 → 로컬 테스트(python payment_agent.py) → project scaffold → 배포 → 호출
```

> **Testnet 전용입니다.** 모든 코드는 [faucet.circle.com](https://faucet.circle.com/)에서 무료 USDC를 받아 Base Sepolia를 사용합니다.

### High-Level 아키텍처

![아키텍처](images/architecture.png)


## 사전 요구 사항

- Tutorial 00 완료(`.env`에 payment manager, instrument 등 설정)
- Tutorial 01 완료(로컬 agent + plugin flow 이해)
- Wallet에 testnet USDC 입금 완료
- Python 3.10+
- Node.js 20+(AgentCore CLI용)
- [AWS CDK](https://docs.aws.amazon.com/cdk/v2/guide/getting_started.html) 설치
- AWS CLI 구성 완료(`aws configure`)

## 1단계: AgentCore CLI 설치

[`@aws/agentcore`](https://github.com/aws/agentcore-cli) CLI는 agent project를 scaffold하고 AgentCore Runtime에 배포 및 호출합니다. Node.js 20+가 필요합니다.

In [ ]:
!npm install -g @aws/agentcore

In [ ]:
!agentcore --version

## 2단계: Python Dependency 설치

private wheel(payments SDK)과 agent framework package를 설치합니다. 로컬 테스트에 필요하며 deployment package에도 함께 bundle됩니다.

In [ ]:
%pip install -r requirements.txt --quiet

## 3단계: AWS Credentials 검증

계속 진행하기 전에 AWS identity와 region을 확인합니다.

In [ ]:
import os
import boto3

# 이름이 지정된 AWS profile을 사용하려면 주석 해제
# os.environ['AWS_PROFILE'] = '<your-profile>'

session = boto3.Session()
identity = session.client("sts").get_caller_identity()
account_id = identity["Account"]
print(f"Authenticated as: {identity['Arn']}")
print(f"Account: {account_id}")
print(f"Region: {session.region_name}")

## 4단계: Payment Config Load

Tutorial 00의 `.env`에서 resource ID를 load합니다. 배포된 agent를 호출할 때 invocation payload로 이 값을 전달합니다.

> **참고:** 이 튜토리얼에서는 편의를 위해 `.env`를 사용합니다. payment credentials는 [AWS Secrets Manager](https://docs.aws.amazon.com/secretsmanager/latest/userguide/intro.html)에 저장하고 IAM role 기반 액세스로 가져오세요.

In [ ]:
import sys

sys.path.append("..")

from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env", override=True)

from utils import load_tutorial_env, print_summary

config = load_tutorial_env()

PAYMENT_MANAGER_ARN = config["payment_manager_arn"]
REGION = config["region"]
USER_ID = config["user_id"]

if config.get("multi_provider"):
    PROVIDER = list(config["instruments"].keys())[0]
    INSTRUMENT_ID = config["instruments"][PROVIDER]["instrument_id"]
else:
    INSTRUMENT_ID = config["instrument_id"]
    PROVIDER = config.get("provider_type", "unknown")

print_summary(
    "Payment Config",
    payment_manager_arn=PAYMENT_MANAGER_ARN,
    region=REGION,
    user_id=USER_ID,
    instrument_id=INSTRUMENT_ID,
    provider=PROVIDER,
)

## 5단계: 로컬 테스트

배포하기 전에 agent가 로컬에서 작동하는지 검증합니다. `BedrockAgentCoreApp`은 AgentCore Runtime과 동일한 contract인 `/invocations` 및 `/ping` endpoint를 사용하여 port 8080에서 Uvicorn server를 시작합니다.

**별도의 terminal에서 agent를 시작합니다.**
```bash
cd 00-getting-started/02-deploy-to-agentcore-runtime
export AWS_PROFILE=<your-profile>
python payment_agent.py
```

> **중요:** agent process에서 올바른 credentials와 region을 사용하도록 `AWS_PROFILE`을 설정해야 합니다.

그런 다음 아래 셀을 실행하여 테스트합니다.

In [ ]:
# 5a. Health check — agent 실행 여부 확인
!curl -s http://localhost:8080/ping

In [ ]:
# 5b. 빠른 호출 — /invocations endpoint 작동 확인
import json
import requests

test_payload = {
    "prompt": "Hello, what can you do?",
    "payment_manager_arn": PAYMENT_MANAGER_ARN,
    "user_id": USER_ID,
    "payment_session_id": "test-local-session",
    "payment_instrument_id": INSTRUMENT_ID,
}

resp = requests.post("http://localhost:8080/invocations", json=test_payload, timeout=60)
print(json.dumps(resp.json(), indent=2))

In [ ]:
# 5c. 전체 payment 테스트 — 실제 session을 생성하고 paid x402 endpoint 호출
import requests
from bedrock_agentcore.payments import PaymentManager

manager = PaymentManager(payment_manager_arn=PAYMENT_MANAGER_ARN, region_name=REGION)

local_session = manager.create_payment_session(
    user_id=USER_ID,
    limits={"maxSpendAmount": {"value": "0.50", "currency": "USD"}},
    expiry_time_in_minutes=60,
)
local_session_id = local_session["paymentSessionId"]
print(f"Session created: {local_session_id} (budget: $0.50)")

paid_payload = {
    "prompt": "Access this paid weather API and tell me what data you get back: https://x402-test.genesisblock.ai/api/weather Report the weather data and how much it cost.",
    "payment_manager_arn": PAYMENT_MANAGER_ARN,
    "user_id": USER_ID,
    "payment_session_id": local_session_id,
    "payment_instrument_id": INSTRUMENT_ID,
}

print("Invoking local agent with paid endpoint...")
resp = requests.post("http://localhost:8080/invocations", json=paid_payload, timeout=120)
print(json.dumps(resp.json(), indent=2))

로컬 테스트를 통과했습니다. agent를 중지하고(terminal에서 `Ctrl+C`) cloud 배포를 계속합니다.

## 6단계: AgentCore Project Scaffold

`agentcore create`는 CLI 배포에 필요한 project 구조(CDK infra, config 파일, app 디렉터리)를 생성합니다. 그런 다음 payment agent 코드와 vendored wheel을 복사합니다.

In [ ]:
# 6a. project scaffold 생성

if not os.path.exists("PaymentAgent"):
    !agentcore create --name PaymentAgent --framework Strands --protocol HTTP --model-provider Bedrock --memory none
else:
    print("PaymentAgent/ already exists — skipping create")

In [ ]:
# 6b. agent 코드를 project에 복사
!cp payment_agent.py PaymentAgent/app/PaymentAgent/main.py

print("Agent code copied:")
!ls PaymentAgent/app/PaymentAgent/

In [ ]:
# 6c. dependency로 pyproject.toml 업데이트
pyproject_content = """[project]
name = "payment-agent"
version = "0.1.0"
requires-python = ">=3.10"
dependencies = [
    "bedrock-agentcore[strands-agents]>=1.9.0",
    "boto3>=1.43.5",
    "strands-agents>=1.0.0",
    "strands-agents-tools>=0.2.0",
    "python-dotenv>=1.0.0",
]
"""
with open("PaymentAgent/app/PaymentAgent/pyproject.toml", "w") as f:
    f.write(pyproject_content)

# 오래된 lock 파일이 있으면 제거
lock_file = "PaymentAgent/app/PaymentAgent/uv.lock"
if os.path.exists(lock_file):
    os.remove(lock_file)

print("pyproject.toml updated")

## 7단계: AgentCore Runtime에 배포

CLI가 Bedrock 모델 + observability 권한이 있는 execution role을 자동으로 생성합니다. 배포 후 해당 role에 payment 권한을 추가합니다.

첫 배포에는 약 2~3분이 걸립니다.

**비용 안내:** 이 배포에서는 Lambda 함수, CloudWatch log group, API Gateway endpoint를 포함한 유료 AWS 리소스를 생성합니다. Bedrock 모델 호출에는 request별 요금이 발생합니다. 지속적인 비용을 방지하려면 작업을 마친 후 리소스 정리 섹션을 실행하세요.

In [ ]:
!cd PaymentAgent && agentcore deploy -y

In [ ]:
# 배포 상태 검증
!cd PaymentAgent && agentcore status

### 자동 생성된 Execution Role에 Payment 권한 추가

CLI가 Bedrock + CloudWatch 권한이 있는 role을 생성했습니다. plugin에서 `ProcessPayment`를 호출할 수 있도록 payment data-plane 권한을 추가합니다.

In [ ]:
import json

iam = boto3.client("iam")

# agentcore deploy에서 생성한 execution role 찾기
roles = iam.list_roles(MaxItems=200)["Roles"]
runtime_roles = [r["RoleName"] for r in roles if "PaymentAgent" in r["RoleName"] and "Execution" in r["RoleName"]]
if not runtime_roles:
    # fallback: 이름에 PaymentAgent가 포함된 모든 role
    runtime_roles = [r["RoleName"] for r in roles if "PaymentAgent" in r["RoleName"]]
assert runtime_roles, "No PaymentAgent role found — check agentcore deploy output"
RUNTIME_ROLE_NAME = runtime_roles[0]

# payment 권한 추가
payment_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:ProcessPayment",
                "bedrock-agentcore:GetPaymentInstrument",
                "bedrock-agentcore:ListPaymentInstruments",
                "bedrock-agentcore:GetPaymentInstrumentBalance",
                "bedrock-agentcore:GetPaymentSession",
                "bedrock-agentcore:GetResourcePaymentToken",
            ],
            "Resource": f"arn:aws:bedrock-agentcore:{REGION}:{account_id}:payment-manager/*",
        }
    ],
}

iam.put_role_policy(
    RoleName=RUNTIME_ROLE_NAME,
    PolicyName="PaymentDataPlaneAccess",
    PolicyDocument=json.dumps(payment_policy),
)

print(f"Added payment permissions to: {RUNTIME_ROLE_NAME}")

In [ ]:
# 이후 튜토리얼(04, 05 등)을 위해 Runtime ARN을 가져와 .env에 저장
import re
from utils import update_env_file

status_output = !cd PaymentAgent && agentcore status --json
raw_text = "\n".join(status_output)
match = re.search(r'arn:aws:bedrock-agentcore:[^\s"]+', raw_text)
assert match, "Could not find Runtime ARN in agentcore status output — is the agent deployed?"
AGENT_RUNTIME_ARN = match.group(0)

update_env_file({"AGENT_RUNTIME_ARN": AGENT_RUNTIME_ARN})
print(f"Runtime ARN: {AGENT_RUNTIME_ARN}")
print("Saved to .env")

## 8단계: 배포된 Agent 호출

새 payment session을 생성한 후(app backend에서 budget 제어) CLI를 통해 배포된 agent를 호출합니다. 

### Payment Flow Sequence

![Payment Flow](images/payment_flow.png)


In [ ]:
# $0.50 budget으로 새 session 생성
fresh_session = manager.create_payment_session(
    user_id=USER_ID,
    limits={"maxSpendAmount": {"value": "0.50", "currency": "USD"}},
    expiry_time_in_minutes=60,
)
fresh_session_id = fresh_session["paymentSessionId"]
print(f"Session: {fresh_session_id} (budget: $0.50, expiry: 60 min)")

In [ ]:
# invocation payload 구성
invoke_payload = json.dumps(
    {
        "prompt": "Access this paid weather API and tell me what data you get back: https://x402-test.genesisblock.ai/api/weather Report the weather data and how much it cost.",
        "payment_manager_arn": PAYMENT_MANAGER_ARN,
        "user_id": USER_ID,
        "payment_session_id": fresh_session_id,
        "payment_instrument_id": INSTRUMENT_ID,
    }
)

print(f"Invoking deployed agent with session {fresh_session_id}...")

In [ ]:
# CLI를 통해 배포된 agent 호출
!cd PaymentAgent && agentcore invoke '{invoke_payload}'

## 9단계: Session 지출 검증

agent의 payment에서 $0.50 budget 중 얼마를 사용했는지 확인합니다.

In [ ]:
session_info = manager.get_payment_session(
    user_id=USER_ID,
    payment_session_id=fresh_session_id,
)

available = session_info.get("availableLimits", {}).get("availableSpendAmount", {})
budget = session_info.get("limits", {}).get("maxSpendAmount", {})

print_summary(
    "Post-Invocation Session",
    session_id=fresh_session_id,
    budget_limit=f"${budget.get('value', 'N/A')} {budget.get('currency', '')}",
    remaining=f"${available.get('value', 'N/A')} {available.get('currency', '')}",
    spent=f"${float(budget.get('value', 0)) - float(available.get('value', 0)):.4f} USD"
    if available.get("value")
    else "N/A",
)

## Observability

AgentCore Runtime은 CloudWatch로 trace와 log를 자동 전송합니다.

In [ ]:
print(
    f"GenAI Dashboard: https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#gen-ai-observability/agent-core"
)
print("Stream logs:     cd PaymentAgent && agentcore logs")

> **경고:** 다음 명령은 AgentCore Runtime 배포와 로컬 project 파일을 영구적으로 삭제합니다. 계속 진행하기 전에 수정한 코드를 backup하세요.

## 리소스 정리

배포된 Runtime과 관련 AWS 리소스를 제거합니다.

**Payment session** — 이 튜토리얼에서 생성한 session은 60 minutes 후 자동으로 만료됩니다. 수동으로 정리할 필요가 없습니다.

In [ ]:
# AgentCore Runtime stack 제거
!cd PaymentAgent && agentcore remove all -y

In [ ]:
# scaffold된 project 디렉터리 제거
import shutil

if os.path.exists("PaymentAgent"):
    shutil.rmtree("PaymentAgent")
    print("Removed PaymentAgent/ directory")

print("Payment stack cleanup: run the cleanup cell in Tutorial 00")

## 요약

payment-enabled agent를 AgentCore Runtime에 배포했습니다.

| 단계 | 작업 | 방법 |
|------|------|-----|
| 로컬 테스트 | 배포 전 agent 작동 검증 | `python payment_agent.py` + curl |
| Scaffold | CLI용 project 구조 생성 | `agentcore create --name PaymentAgent` |
| 배포 | CDK를 통해 AWS에 package + 배포 | `agentcore deploy` |
| 호출 | 배포된 agent 호출 | `agentcore invoke '{...}'` |
| 리소스 정리 | 모든 리소스 제거 | `agentcore remove all` + `agentcore deploy` |

배포된 agent의 동작은 다음과 같습니다.
- invocation payload에서 모든 payment context 수신(stateless)
- plugin이 402 → ProcessPayment 자동 처리
- session budget 초과 불가(server-side에서 적용)
- Tutorial 00에서 구성한 지원 wallet provider 모두에서 작동

### 다음: Tutorial 03 — User Onboarding 및 Wallet Funding